# 01 Data preparation

Builds the district-day panel and the feature matrix from the frozen inputs, checks the panel
against its expected counts, and summarises the splits and the per-year rates. All logic lives in
`src/`; this notebook calls it and shows the results.

- **Inputs:** `data/raw/` (BIPAD incident snapshot and geography), `data/weather/era5-daily-districts.csv`,
  `data/processed/district-terrain.csv`
- **Outputs:** `data/processed/district-day-panel.parquet`, `hydromet-incidents.parquet` and
  `feature-matrix.parquet`
- **Runtime:** under a minute

Panel row: one district and one issue day t (1 April to 28 October). Primary target `y = 1` if the
district has a damaging Flood, Landslide, Heavy Rainfall or Lightning incident dated t+1 to t+3.

## Setup

In [1]:
import sys
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))


from src import config as C
from src import panel, weather
from src.features import FEATURES_A, FEATURES_B, load_all
from src.freeze import MATRIX_PATH

## 1. Build the panel and the incidents table

In [2]:
incidents, P = panel.build()
print(
    f"incidents of the four hazards: {len(incidents):,} (damaging {int(incidents['damaging'].sum()):,})"
)
print(
    f"panel: {P.shape[0]:,} rows x {P.shape[1]} columns, {P['district'].nunique()} districts, seasons {P['season'].min()} to {P['season'].max()}"
)
print("matches the expected counts:", panel.check(P))

incidents of the four hazards: 16,790 (damaging 7,994)
panel: 156,849 rows x 20 columns, 77 districts, seasons 2017 to 2026
matches the expected counts: True


## 2. Splits (time-based, never random)

In [3]:
panel.split_table(P)

,rows,positives,rate
split,,,
train,97482,6482,6.65
validate,16247,722,4.44
test,32494,2749,8.46
check,10626,614,5.78


## 3. Per-year positive rates (%)

Any incident and damaging are same-day rates; the targets look 3 days ahead.

In [4]:
by_year = P.groupby("season").agg(
    rows=("y", "size"),
    any_incident=("day_any", "mean"),
    damaging=("day_damaging", "mean"),
    y=("y", "mean"),
    y_fatal=("y_fatal", "mean"),
)
by_year[["any_incident", "damaging", "y", "y_fatal"]] *= 100
by_year.round(2)

,rows,any_incident,damaging,y,y_fatal
season,,,,,
2017,16247,2.87,2.40,5.91,3.37
2018,16247,3.88,2.45,6.81,2.09
2019,16247,4.50,2.50,6.54,2.80
2020,16247,5.16,3.11,8.18,2.98
2021,16247,4.57,2.68,7.05,2.99
2022,16247,3.80,2.02,5.41,2.22
2023,16247,4.52,1.56,4.44,1.45
2024,16247,8.08,3.45,9.02,3.35
2025,16247,7.69,2.85,7.90,2.20


## 4. Target rates per hazard, 2017 to 2025 (%)

In [5]:
main = P[P["season"].isin(C.SEASONS)]
(100 * main[panel.TARGETS].mean()).round(2).to_frame("positive rate (%)")

,positive rate (%)
y,6.81
y_fatal,2.60
y_rain,4.35
y_flood,1.00
y_landslide,2.19
y_heavy_rain,1.67
y_lightning,2.60


## 5. Frozen weather and terrain inputs

ERA5 daily weather at the district centroids and the terrain statistics are committed in
`data/weather/` and `data/processed/`, so no network access is needed here.

In [6]:
W = weather.load_weather()
T = weather.load_terrain()
print(
    f"ERA5: {len(W):,} district-days, {W['district'].nunique()} districts, "
    f"{W['date'].min():%Y-%m-%d} to {W['date'].max():%Y-%m-%d}"
)
print(f"terrain: {len(T)} districts, columns {list(T.columns[1:])}")

ERA5: 201,971 district-days, 77 districts, 2016-03-02 to 2026-08-31
terrain: 77 districts, columns ['elev_mean', 'elev_relief', 'share_above_1000m', 'share_above_3000m']


## 6. Feature matrix

Set A is known at day t; set B adds observed rain for t+1 to t+3 (`future_*`, a perfect forecast).
The matrix is saved for notebooks 02 to 05 (the same file `uv run python -m src.freeze` writes).

In [7]:
F = load_all()
F.to_parquet(MATRIX_PATH, index=False)
print(f"feature table {F.shape}; set A {len(FEATURES_A)} features, set B {len(FEATURES_B)}")
F[FEATURES_B].notna().mean().mul(100).round(1).to_frame("% non-missing").T

feature table (156849, 51); set A 29 features, set B 32


,rain_1d,rain_3d,rain_7d,rain_14d,rain_30d,rain_api,rain_hours_1d,tmax_1d,rain_3d_pct,doy_sin,...,recent_heavy_rain_7d,recent_lightning_7d,te_dam,te_flood,te_landslide,te_heavy_rain,te_lightning,future_rain_3d,future_rain_max_1d,future_rain_hours_3d
% non-missing,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
